# Prompt Engineering

This notebook explores and compares different prompt engineering techniques against a small, locally-run LLM (`Llama-3.2-3B-Instruct`, 4-bit quantized via `mlx-lm`).

Two task families are used to probe how prompt structure affects output quality:

1. **Sentiment/aspect classification** (a hotel review that mixes praise and criticism) — used to compare **zero-shot**, **one-shot**, and **few-shot** prompting.
2. **Multi-step arithmetic word problems** — used to compare a **naive** direct-answer prompt against **Chain-of-Thought (CoT)** prompting, and finally to demonstrate **self-consistency** (sampling multiple CoT reasoning paths and taking a majority vote over the final answers).

Each prompt cell below includes a comment stating the expected correct answer, so the model's actual output can be checked at a glance.


In [ ]:
# mlx-lm runs quantized models natively on Apple Silicon's Metal GPU,
# avoiding the CUDA-only bitsandbytes 4-bit path that transformers relies on.
from mlx_lm import load, stream_generate
from mlx_lm.sample_utils import make_sampler

In [ ]:
# Llama-3.2-3B-Instruct-4bit is a current mlx-community checkpoint (proper
# safetensors weights), light enough (~1.9GB) for quick prompt-engineering runs.
model, tokenizer = load("mlx-community/Llama-3.2-3B-Instruct-4bit")

In [27]:
def generate(input_text: str, temperature: float = 0.1, max_new_tokens: int = 10000) -> str:
    """Stream an assistant response from the MLX-quantized Llama-3.2 model.

    Args:
        input_text: The user's message to prompt the model with.
        temperature: Sampling temperature passed to the MLX sampler; 0 is greedy decoding.
        max_new_tokens: Maximum number of tokens to generate.

    Returns:
        The full generated response text.
    """
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": input_text}],
        add_generation_prompt=True,
    )
    sampler = make_sampler(temp=temperature)
    response_text = ""
    for chunk in stream_generate(model, tokenizer, prompt, max_tokens=max_new_tokens, sampler=sampler):
        print(chunk.text, end="", flush=True)
        response_text += chunk.text
    return response_text

## 1. Zero-shot vs. one-shot vs. few-shot classification

The same review — one with clearly mixed sentiment — is classified three times with an increasing number of in-context examples:

- **Zero-shot**: only the task instructions are given, no examples.
- **One-shot**: a single labeled example precedes the query.
- **Few-shot**: four labeled examples (covering Positive, Negative, and Mixed) precede the query.

The goal is to see whether adding examples helps the model correctly identify the harder "Mixed" category instead of defaulting to Positive or Negative.


In [28]:
zero_shot_prompt = """
Classify the following review as Positive, Negative, Neutral or Mixed.
Mixed means the review expresses both praise and criticism in comparable amounts.
Return only the classification, nothing else.

Text: The staff was incredibly friendly and checked us in fast, but the room reeked of smoke and the AC never worked the entire stay.
Answer:
"""

# Expected answer: Mixed (praise for the staff/check-in, criticism of the smoky room and broken AC)
zero_shot_output = generate(zero_shot_prompt, temperature=0.0, max_new_tokens=5)

Mixed

In [29]:
one_shot_prompt = """
Classify the following review as Positive, Negative, Neutral or Mixed.
Mixed means the review expresses both praise and criticism in comparable amounts.
Return only the classification, nothing else.

Text: The food was delicious and beautifully plated, but we waited over an hour for our main course.
Answer: Mixed

Text: The staff was incredibly friendly and checked us in fast, but the room reeked of smoke and the AC never worked the entire stay.
Answer:
"""

# Expected answer: Mixed (praise for the staff/check-in, criticism of the smoky room and broken AC)
one_shot_output = generate(one_shot_prompt, temperature=0.0, max_new_tokens=5)

Mixed

In [30]:
few_shot_prompt = """
Classify the following review as Positive, Negative, Neutral or Mixed.
Mixed means the review expresses both praise and criticism in comparable amounts.
Return only the classification, nothing else.

Text: The food was delicious and beautifully plated, but we waited over an hour for our main course.
Answer: Mixed

Text: Check-in was smooth and the pool area was spotless.
Answer: Positive

Text: Oh sure, a "complimentary" breakfast that ran out of everything by 8am, truly five-star service.
Answer: Negative

Text: The concierge booked us a great restaurant, though the elevator was out of service for two of our three days.
Answer: Mixed

Text: The staff was incredibly friendly and checked us in fast, but the room reeked of smoke and the AC never worked the entire stay.
Answer:
"""

# Expected answer: Mixed (praise for the staff/check-in, criticism of the smoky room and broken AC)
few_shot_output = generate(few_shot_prompt, temperature=0.0, max_new_tokens=5)

Mixed

## 2. Naive prompting vs. Chain-of-Thought (CoT)

A multi-step arithmetic word problem is used to compare:

- **Naive prompting**: the model is asked to jump straight to the final number, with no intermediate reasoning.
- **Chain-of-Thought prompting**: the model is explicitly instructed to think step by step and report the running total after each event, before giving the final answer.

CoT prompting typically improves accuracy on multi-step problems because it gives the model room to work through intermediate results instead of trying to compute everything in a single forward pass.


In [31]:
naive_prompt = """
A bakery baked 3 trays with 8 cookies each. They sold half of all their cookies in the morning.
In the afternoon they baked 2 more trays of 8 cookies each, but 5 cookies from that new batch came out burnt and were thrown away.
How many cookies does the bakery have now? Answer with only the final number, no explanation.
"""

# Expected answer: 23 (3*8=24 baked, 12 sold -> 12 left; +2*8=16 baked, -5 burnt -> +11; 12+11=23)
naive_output = generate(naive_prompt, temperature=0.0, max_new_tokens=10)

44

In [32]:
CoT_prompt = """
A bakery baked 3 trays with 8 cookies each. They sold half of all their cookies in the morning.
In the afternoon they baked 2 more trays of 8 cookies each, but 5 cookies from that new batch came out burnt and were thrown away.
How many cookies does the bakery have now?

Let's think step by step, stating the running total after each event before moving to the next.
"""

# Expected answer: 23 (3*8=24 baked, 12 sold -> 12 left; +2*8=16 baked, -5 burnt -> +11; 12+11=23)
CoT_output = generate(CoT_prompt, temperature=0.0, max_new_tokens=300)

Let's break it down step by step.

Initially, the bakery baked 3 trays with 8 cookies each, so they had a total of:
3 trays * 8 cookies/tray = 24 cookies

They sold half of all their cookies in the morning. Half of 24 cookies is:
24 cookies / 2 = 12 cookies

So, after selling 12 cookies in the morning, the bakery had:
24 cookies - 12 cookies = 12 cookies

In the afternoon, they baked 2 more trays of 8 cookies each, which is a total of:
2 trays * 8 cookies/tray = 16 cookies

However, 5 cookies from that new batch came out burnt and were thrown away. So, the bakery now has:
12 cookies + 16 cookies - 5 cookies = 23 cookies

The bakery now has 23 cookies.

## 3. Self-consistency

Self-consistency extends CoT by sampling several independent reasoning paths (with non-zero temperature, so the wording and intermediate steps vary between samples) and taking a **majority vote** over each path's final numeric answer. This trades extra inference calls for a higher chance that noise in any single reasoning chain gets outvoted by the consensus answer.

The classic "trains and a bird" puzzle is used here: instead of tracking the bird's back-and-forth trips, the trick is to compute the time until the trains meet and multiply by the bird's speed.


In [33]:
import re
from collections import Counter


def majority_vote(responses: list[str], pattern: str = r"-?\d+") -> str:
    """Pick the most common final numeric answer across several model responses.

    Args:
        responses: Raw model completions, each expected to end with a numeric answer.
        pattern: Regex used to find numeric tokens within each response.

    Returns:
        The numeric answer (as a string) that appears most often across responses.
    """
    answers = []
    for response in responses:
        matches = re.findall(pattern, response)
        if matches:
            answers.append(matches[-1])
    return Counter(answers).most_common(1)[0][0]


# Expected answer: 270 miles (trains close a 300-mile gap at 60+40=100 mph, so they meet
# after 3 hours; the bird flies for that same 3 hours at 90 mph -> 90*3=270)
sc_prompt = """
Train A leaves City A at 60 mph heading straight toward City B, which is 300 miles away.
At the same moment, Train B leaves City B heading straight toward City A at 40 mph.
Also at that same moment, a bird starts flying back and forth between the two trains at 90 mph,
turning around instantly each time it reaches a train, until the trains meet.
How many total miles does the bird fly before the trains meet?

Let's think step by step.
"""

In [34]:
NUM_SAMPLES = 5

sc_responses = [
    generate(sc_prompt, temperature=0.7, max_new_tokens=350)
    for _ in range(NUM_SAMPLES)
]

sc_answer = majority_vote(sc_responses)
print(f"\n\nSelf-consistency answer over {NUM_SAMPLES} samples: {sc_answer} miles")

To solve this problem, we need to follow the steps below:

1. Calculate the combined speed of Train A and Train B:
Since Train A is heading towards City B at 60 mph and Train B is heading towards City A at 40 mph, their combined speed is 60 + 40 = 100 mph.

2. Calculate the time it takes for Train A and Train B to meet:
The distance between City A and City B is 300 miles, and their combined speed is 100 mph. So, the time it takes for them to meet is 300 / 100 = 3 hours.

3. Calculate the speed of the bird:
The bird is flying back and forth between the two trains at 90 mph.

4. Calculate the distance the bird flies per hour:
Since the bird is constantly turning around, its effective speed is half of its actual speed. So, the distance the bird flies per hour is 90 / 2 = 45 miles per hour.

5. Calculate the total distance the bird flies in 3 hours:
Since the bird flies 45 miles per hour, and the time it takes for the trains to meet is 3 hours, the total distance the bird flies before the 

## Summary

This notebook worked through three prompt engineering techniques, moving from simplest to most expensive:

- **In-context examples (zero/one/few-shot)** shape how the model interprets an ambiguous task, such as recognizing "Mixed" sentiment instead of collapsing it to Positive or Negative.
- **Chain-of-Thought prompting** improves multi-step arithmetic reasoning by asking the model to externalize intermediate steps rather than jumping straight to a final number.
- **Self-consistency** further improves reliability on reasoning tasks by sampling multiple CoT paths at non-zero temperature and taking a majority vote over their final answers, trading extra compute for robustness against any single flawed reasoning chain.

Each technique trades off prompt/inference cost against output quality, and the right choice depends on task difficulty and how much latency/compute budget is available.
